In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from src.utils.paths import load_paths
paths = load_paths()

## 0. Imports & Config

This notebook compares multiple SVM hyperparameter configurations on the
few-shot classification pipeline using the **direct config comparison** approach.

**Method:**
1. Define 6 candidate (C, gamma, kernel) configurations
2. Run full 100 MC sweeps at every K for each config (same seeds → identical splits)
3. Compare macro accuracy curves, per-species accuracy, and confusion matrices

See `docs/HP_methods.md` for details on this approach vs. the old GridSearchCV method.

In [ ]:
# === Imports & Config ===

import json, random, hashlib, os, time, shutil
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

from bioclip.predict import BaseClassifier

# ---- Paths ----
CLASS_NAMES = paths['processed_dir'] / 'class_names_cleaned.json'
DATA        = paths['processed_dir'] / 'final_data_cleaned.json'
EMB_CACHE   = paths['processed_dir'] / 'emb_cache'

# ---- Experiment config ----
SHOTS      = [1, 3, 5, 10, 25, 35]
RUNS       = 100
BASE_SEED  = 42
MIN_TOTAL  = 50

# ---- New configs to run ----
CONFIGS = {
    'G_finer':      {'C': 10,  'gamma': 0.0005, 'kernel': 'rbf'},
    'H_wider':      {'C': 10,  'gamma': 0.005,  'kernel': 'rbf'},
    'I_tiny_gamma': {'C': 10,  'gamma': 0.0001, 'kernel': 'rbf'},
    'J_logreg':     {'type': 'logreg'},  # special case
}

# ---- Previous run to merge with ----
PREV_RUN = Path('results/svm_hp/runs/260405_165026')

# ---- Results directory (timestamped) ----
HP_RESULTS = paths.get('svm_hp_results_dir', paths['results_dir'] / 'svm_hp')
if isinstance(HP_RESULTS, str):
    HP_RESULTS = paths['results_dir'].parent / HP_RESULTS
HP_RESULTS = Path(HP_RESULTS)

RUN_ID = datetime.now().strftime("%y%m%d_%H%M%S")
RUN_DIR = HP_RESULTS / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Create per-config output dirs
CONFIG_DIRS = {}
for cfg_name in CONFIGS:
    cfg_dir = RUN_DIR / cfg_name
    (cfg_dir / 'analysis').mkdir(parents=True, exist_ok=True)
    (cfg_dir / 'plots').mkdir(parents=True, exist_ok=True)
    CONFIG_DIRS[cfg_name] = cfg_dir

# Comparison dir (will merge old + new)
COMP_DIR = RUN_DIR / 'comparison'
(COMP_DIR / 'analysis').mkdir(parents=True, exist_ok=True)
(COMP_DIR / 'plots').mkdir(parents=True, exist_ok=True)

with open(HP_RESULTS / "latest_run.txt", "w") as f:
    f.write(str(RUN_DIR))

# ---- Seed derivation (same as notebook 03) ----
def seed_for(k, run_id):
    tag = f"{BASE_SEED}_{k}_{run_id}"
    return int(hashlib.sha256(tag.encode()).hexdigest()[:8], 16) % (2**31)

# ---- BioCLIP feature extractor ----
device = "cuda" if torch.cuda.is_available() else "cpu"
BC = BaseClassifier(device=device)

print(f"Device: {device}")
print(f"Run ID: {RUN_ID}")
print(f"Results: {RUN_DIR}")
print(f"Previous run: {PREV_RUN}")
print(f"\nNew configurations to run:")
for name, params in CONFIGS.items():
    if params.get('type') == 'logreg':
        print(f"  {name}: LogisticRegression (L2, saga, balanced, C auto-range)")
    else:
        print(f"  {name}: C={params['C']}, gamma={params['gamma']}, kernel={params['kernel']}")

## 1. Load Data & Build Specimen Index

Identical to notebook 03.

In [ ]:
# === Block 1 — Load Classes and Data -> Build Specimen Index ===

with open(CLASS_NAMES, "r") as f:
    SPECIES_LIST = set(json.load(f))

with open(DATA, "r") as f:
    data = json.load(f)

by_species = defaultdict(lambda: defaultdict(dict))
for r in data:
    sp = r["true_label"]
    if sp not in SPECIES_LIST:
        continue
    sid  = r["sample_id"]
    view = str(r["view"]).strip().lower()
    by_species[sp][sid][view] = r["image_path"]

# Keep only specimens with BOTH views
for sp in list(by_species.keys()):
    for sid in list(by_species[sp].keys()):
        views = by_species[sp][sid]
        if not ("dorsal" in views and "ventral" in views):
            del by_species[sp][sid]
    if not by_species[sp]:
        del by_species[sp]

# Enforce minimum specimen count
for sp in list(by_species.keys()):
    if len(by_species[sp]) < MIN_TOTAL:
        del by_species[sp]

included_species = sorted(by_species.keys())
SPECIES_N = {sp: len(by_species[sp]) for sp in included_species}
CLASS_ORDER = sorted(included_species, key=lambda sp: SPECIES_N[sp], reverse=True)

n_specimens = sum(len(smap) for smap in by_species.values())
print(f"Included species: {len(included_species)}")
print(f"Total usable specimens: {n_specimens}")
for sp in CLASS_ORDER:
    print(f"  {sp}: {SPECIES_N[sp]} specimens")

In [ ]:
# === Blocks 2-4 — Split, Embedding, and Train/Predict helpers ===

# --- Embedding helpers (Block 3) ---
def _cache_fp(img_path: str) -> Path:
    h = hashlib.sha256(img_path.encode("utf-8")).hexdigest()[:24]
    return EMB_CACHE / f"{h}.npy"

def embed_image(img_path: str) -> np.ndarray:
    fp = _cache_fp(img_path)
    if fp.exists():
        return np.load(fp)
    pil = Image.open(img_path).convert("RGB")
    vec = BC.create_image_features([pil], normalize=True).cpu().numpy()[0]
    np.save(fp, vec)
    return vec

def specimen_vec(rec: dict) -> np.ndarray:
    z_d = embed_image(rec["dorsal"])
    z_v = embed_image(rec["ventral"])
    return 0.5 * (z_d + z_v)

# --- Split helper (Block 2) ---
def split_once(by_species, species_list, k, seed):
    rng = random.Random(seed)
    train_pairs, test_pairs = [], []
    for sp in species_list:
        sids = list(by_species[sp].keys())
        rng.shuffle(sids)
        tr, te = sids[:k], sids[k:]
        assert len(tr) == k, f"{sp}: needs {k} train, found {len(tr)}"
        assert len(te) >= 1, f"{sp}: needs at least 1 test specimen"
        train_pairs.extend([(sp, sid) for sid in tr])
        test_pairs.extend([(sp, sid) for sid in te])
    return train_pairs, test_pairs

def split_for_run(by_species, species_list, k, run_id):
    return split_once(by_species, species_list, k=k, seed=seed_for(k, run_id))

# --- Build X/y (Block 4) ---
def build_xy(by_species, pairs):
    X, y, ids = [], [], []
    for sp, sid in pairs:
        rec = by_species[sp][sid]
        X.append(specimen_vec(rec))
        y.append(sp)
        ids.append(sid)
    return np.stack(X), np.array(y), ids

print("Helpers loaded.")

## 2. Full Monte Carlo Sweep — All Configurations

Run 100 MC splits at every K for each of the 5 configs.
Same seeds across configs guarantee identical train/test splits.

In [ ]:
# === Full Monte Carlo Sweep — New Configurations ===

from tqdm.auto import tqdm

all_config_predictions = {}  # config_name -> DataFrame

for config_name, params in CONFIGS.items():
    is_logreg = params.get('type') == 'logreg'
    if is_logreg:
        print(f"\n{'='*60}")
        print(f"Config {config_name}: LogisticRegression (L2, saga, balanced)")
        print(f"{'='*60}")
    else:
        print(f"\n{'='*60}")
        print(f"Config {config_name}: C={params['C']}, gamma={params['gamma']}, kernel={params['kernel']}")
        print(f"{'='*60}")
    
    pred_rows = []
    cfg_dir = CONFIG_DIRS[config_name]
    
    for shots in SHOTS:
        pbar = tqdm(range(RUNS), desc=f"  {config_name} {shots}-shot", leave=True)
        for run_id in pbar:
            try:
                seed = seed_for(shots, run_id)
                train_pairs, test_pairs = split_for_run(
                    by_species, included_species, k=shots, run_id=run_id
                )
                
                Xtr, ytr, _ = build_xy(by_species, train_pairs)
                Xte, yte, te_ids = build_xy(by_species, test_pairs)
                
                if is_logreg:
                    clf = Pipeline([
                        ("scaler", StandardScaler()),
                        ("clf", LogisticRegression(
                            max_iter=1000,
                            class_weight="balanced",
                            penalty="l2",
                            solver="saga",
                            random_state=seed))
                    ])
                else:
                    clf = Pipeline([
                        ("scaler", StandardScaler()),
                        ("clf", SVC(C=params['C'], gamma=params['gamma'],
                                    kernel=params['kernel'],
                                    class_weight="balanced", probability=True,
                                    random_state=seed))
                    ])
                
                clf.fit(Xtr, ytr)
                yhat = clf.predict(Xte)
                probs = clf.predict_proba(Xte)
                conf = probs.max(axis=1)
                
                overall_acc = accuracy_score(yte, yhat)
                macro_acc = balanced_accuracy_score(yte, yhat)
                
                for i, (sp_true, sid) in enumerate(zip(yte, te_ids)):
                    pred_rows.append({
                        'config': config_name,
                        'shots': shots,
                        'run_id': run_id,
                        'sample_id': sid,
                        'species_true': sp_true,
                        'species_pred': yhat[i],
                        'correct': int(sp_true == yhat[i]),
                        'confidence': float(conf[i]),
                        'macro_acc': macro_acc,
                        'overall_acc': overall_acc
                    })
                
                pbar.set_postfix(macro=f"{macro_acc:.3f}")
            
            except Exception as e:
                print(f"  [ERROR] {config_name} shots={shots} run={run_id}: {e}")
    
    df_config = pd.DataFrame(pred_rows)
    df_config.to_csv(cfg_dir / "predictions.csv", index=False)
    all_config_predictions[config_name] = df_config
    print(f"  -> {len(df_config)} rows saved to {cfg_dir / 'predictions.csv'}")

print(f"\nAll new configs complete.")

## 3. Aggregate Metrics

Compute macro accuracy and per-species accuracy for each config at each K.

In [ ]:
# === Aggregate new configs + load previous run results ===

# --- Aggregate new configs ---
config_summaries = {}
config_species = {}

for config_name, df_cfg in all_config_predictions.items():
    cfg_dir = CONFIG_DIRS[config_name]
    
    shot_rows = []
    for K in SHOTS:
        subset = df_cfg[df_cfg['shots'] == K]
        run_macros = subset.groupby('run_id')['macro_acc'].first()
        run_overalls = subset.groupby('run_id')['overall_acc'].first()
        shot_rows.append({
            'shots': K,
            'mean_macro': run_macros.mean(),
            'std_macro': run_macros.std(),
            'ci95_macro': 1.96 * run_macros.std() / np.sqrt(len(run_macros)),
            'mean_overall': run_overalls.mean(),
            'std_overall': run_overalls.std(),
            'ci95_overall': 1.96 * run_overalls.std() / np.sqrt(len(run_overalls)),
            'n_runs': len(run_macros)
        })
    shot_df = pd.DataFrame(shot_rows)
    shot_df.to_csv(cfg_dir / 'analysis' / 'shot_summary.csv', index=False)
    config_summaries[config_name] = shot_df
    
    sp_rows = []
    for K in SHOTS:
        subset = df_cfg[df_cfg['shots'] == K]
        for sp in CLASS_ORDER:
            sp_data = subset[subset['species_true'] == sp]
            per_run = sp_data.groupby('run_id')['correct'].mean()
            sp_rows.append({
                'shots': K, 'species': sp,
                'acc_mean': per_run.mean(), 'acc_std': per_run.std(),
                'acc_ci95': 1.96 * per_run.std() / np.sqrt(len(per_run)),
                'n_runs': len(per_run)
            })
    sp_df = pd.DataFrame(sp_rows)
    sp_df.to_csv(cfg_dir / 'analysis' / 'per_species_accuracy.csv', index=False)
    config_species[config_name] = sp_df

# --- Load previous run results (A-E from run 260405_165026) ---
PREV_CONFIGS = {
    'A_default':     {'C': 1.0,  'gamma': 'scale', 'kernel': 'rbf'},
    'B_smooth':      {'C': 10,   'gamma': 0.001,   'kernel': 'rbf'},
    'C_tight':       {'C': 100,  'gamma': 'scale', 'kernel': 'rbf'},
    'D_linear':      {'C': 10,   'gamma': 'scale', 'kernel': 'linear'},
    'E_regularized': {'C': 0.1,  'gamma': 'auto',  'kernel': 'rbf'},
}

for prev_name in PREV_CONFIGS:
    prev_summary = PREV_RUN / prev_name / 'analysis' / 'shot_summary.csv'
    prev_species = PREV_RUN / prev_name / 'analysis' / 'per_species_accuracy.csv'
    prev_preds = PREV_RUN / prev_name / 'predictions.csv'
    if prev_summary.exists():
        config_summaries[prev_name] = pd.read_csv(prev_summary)
        config_species[prev_name] = pd.read_csv(prev_species)
        all_config_predictions[prev_name] = pd.read_csv(prev_preds)
        print(f"Loaded previous: {prev_name}")

# Merge all configs for comparison
ALL_CONFIGS = {**PREV_CONFIGS, **CONFIGS}

# Find best config at K=35
best_config = max(config_summaries, key=lambda c: config_summaries[c][config_summaries[c]['shots']==35]['mean_macro'].values[0])

print(f"\n--- All configs (previous + new) ---")
for name in sorted(config_summaries.keys()):
    m35 = config_summaries[name][config_summaries[name]['shots']==35]['mean_macro'].values[0]
    marker = " <-- BEST" if name == best_config else ""
    print(f"  {name}: {m35*100:.1f}%{marker}")

## 4. Per-Config Plots

For each config: macro accuracy curve, per-species accuracy, and combined overlay.

In [ ]:
# === Per-config plots (new configs only) ===

shots_grid = np.array(SHOTS)

for config_name in CONFIGS:
    summary = config_summaries[config_name]
    species = config_species[config_name]
    plots_dir = CONFIG_DIRS[config_name] / 'plots'
    p = CONFIGS[config_name]
    if p.get('type') == 'logreg':
        label = "LogReg (L2, saga, balanced)"
    else:
        label = f"C={p['C']}, gamma={p['gamma']}, {p['kernel']}"
    
    # ── Plot 1: Macro Accuracy ──
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    ax.plot(summary['shots'], summary['mean_macro'], 'o-', linewidth=2, markersize=8)
    ax.fill_between(summary['shots'],
                    summary['mean_macro'] - summary['ci95_macro'],
                    summary['mean_macro'] + summary['ci95_macro'], alpha=0.15)
    for _, row in summary.iterrows():
        ax.annotate(f"{row['mean_macro']*100:.1f}%",
                    (row['shots'], row['mean_macro']),
                    textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=9, fontweight='semibold')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_ylim(0.5, 1.0)
    ax.set_xlabel('Shots per species', fontsize=12)
    ax.set_ylabel('Macro accuracy', fontsize=12)
    ax.set_title(f'Macro Accuracy — {config_name}\n({label}, mean +/- 95% CI)', fontsize=13)
    ax.set_xticks(shots_grid); ax.grid(True, axis='y', alpha=0.25)
    plt.tight_layout()
    plt.savefig(plots_dir / 'macro_accuracy.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    # ── Plot 2: Overall Accuracy ──
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    ax.plot(summary['shots'], summary['mean_overall'], 'o-', linewidth=2, markersize=8, color='#2ca02c')
    ax.fill_between(summary['shots'],
                    summary['mean_overall'] - summary['ci95_overall'],
                    summary['mean_overall'] + summary['ci95_overall'], alpha=0.15, color='#2ca02c')
    for _, row in summary.iterrows():
        ax.annotate(f"{row['mean_overall']*100:.1f}%",
                    (row['shots'], row['mean_overall']),
                    textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=9, fontweight='semibold', color='#2ca02c')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_ylim(0.5, 1.0)
    ax.set_xlabel('Shots per species', fontsize=12)
    ax.set_ylabel('Overall accuracy', fontsize=12)
    ax.set_title(f'Overall Accuracy — {config_name}\n({label}, mean +/- 95% CI)', fontsize=13)
    ax.set_xticks(shots_grid); ax.grid(True, axis='y', alpha=0.25)
    plt.tight_layout()
    plt.savefig(plots_dir / 'overall_accuracy.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    # ── Plot 3: Per-species Accuracy ──
    fig, ax = plt.subplots(figsize=(9.5, 6.5))
    for sp in CLASS_ORDER:
        sp_data = species[species['species'] == sp].set_index('shots').reindex(shots_grid).reset_index()
        y = sp_data['acc_mean'].to_numpy()
        ci = sp_data['acc_ci95'].to_numpy()
        ax.plot(shots_grid, y, marker='o', linewidth=1.5, label=sp, alpha=0.8)
        ax.fill_between(shots_grid, y - ci, y + ci, alpha=0.1, linewidth=0)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xticks(shots_grid); ax.set_ylim(0, 1.05)
    ax.set_xlabel('Shots per species', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'Per-Species Accuracy — {config_name}\n({label}, mean +/- 95% CI)', fontsize=13)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax.grid(True, axis='y', alpha=0.25)
    plt.tight_layout()
    plt.savefig(plots_dir / 'per_species_accuracy.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    # ── Plot 4: Combined (per-species + macro black line) ──
    fig, ax = plt.subplots(figsize=(10, 7))
    for sp in CLASS_ORDER:
        sp_data = species[species['species'] == sp].set_index('shots').reindex(shots_grid).reset_index()
        y = sp_data['acc_mean'].to_numpy()
        ci = sp_data['acc_ci95'].to_numpy()
        ax.plot(shots_grid, y, marker='o', linewidth=1.5, label=sp, alpha=0.8)
        ax.fill_between(shots_grid, y - ci, y + ci, alpha=0.1, linewidth=0)
    ax.plot(summary['shots'], summary['mean_macro'], marker='s', linewidth=2,
            color='black', label='Macro Accuracy (mean)', zorder=10, markersize=5)
    ax.fill_between(summary['shots'],
                    summary['mean_macro'] - summary['ci95_macro'],
                    summary['mean_macro'] + summary['ci95_macro'],
                    alpha=0.1, color='black', linewidth=0)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xticks(shots_grid); ax.set_ylim(0, 1.05)
    ax.set_xlabel('Shots per species', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'Per-Species + Macro Accuracy — {config_name}\n({label}, mean +/- 95% CI)', fontsize=13)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax.grid(True, axis='y', alpha=0.25)
    plt.tight_layout()
    plt.savefig(plots_dir / 'combined_species_macro.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    print(f"[{config_name}] 4 plots saved to {plots_dir}")

In [ ]:
# === Comparison plots: all configs (previous + new) ===

CONFIG_COLORS = {
    'A_default': '#4C72B0',
    'B_smooth': '#E07B39',
    'C_tight': '#55A868',
    'D_linear': '#C44E52',
    'E_regularized': '#8172B2',
    'G_finer': '#CCB974',
    'H_wider': '#64B5CD',
    'I_tiny_gamma': '#DD8452',
    'J_logreg': '#000000',
}

all_names = sorted(config_summaries.keys())

# ── Plot: All configs macro accuracy overlay ──
fig, ax = plt.subplots(figsize=(12, 7))
for config_name in all_names:
    s = config_summaries[config_name]
    color = CONFIG_COLORS.get(config_name, '#999999')
    style = 'o-' if config_name == best_config else 'o--'
    lw = 2.5 if config_name == best_config else 1.5
    alpha = 1.0 if config_name == best_config else 0.7
    ax.plot(s['shots'], s['mean_macro'], style, color=color,
            linewidth=lw, markersize=7, label=config_name, alpha=alpha)
    ax.fill_between(s['shots'],
                    s['mean_macro'] - s['ci95_macro'],
                    s['mean_macro'] + s['ci95_macro'],
                    alpha=0.08, color=color)

ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xticks(shots_grid)
ax.set_ylim(0.5, 1.0)
ax.set_xlabel('Shots per species', fontsize=12)
ax.set_ylabel('Macro accuracy', fontsize=12)
ax.set_title('Macro Accuracy — All Configs\n(mean +/- 95% CI, 100 MC runs)', fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(COMP_DIR / 'plots' / 'all_configs_macro_overlay.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Plot: All configs overall accuracy overlay ──
fig, ax = plt.subplots(figsize=(12, 7))
for config_name in all_names:
    s = config_summaries[config_name]
    color = CONFIG_COLORS.get(config_name, '#999999')
    style = 'o-' if config_name == best_config else 'o--'
    lw = 2.5 if config_name == best_config else 1.5
    alpha = 1.0 if config_name == best_config else 0.7
    ax.plot(s['shots'], s['mean_overall'], style, color=color,
            linewidth=lw, markersize=7, label=config_name, alpha=alpha)
    ax.fill_between(s['shots'],
                    s['mean_overall'] - s['ci95_overall'],
                    s['mean_overall'] + s['ci95_overall'],
                    alpha=0.08, color=color)

ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xticks(shots_grid)
ax.set_ylim(0.5, 1.0)
ax.set_xlabel('Shots per species', fontsize=12)
ax.set_ylabel('Overall accuracy', fontsize=12)
ax.set_title('Overall Accuracy — All Configs\n(mean +/- 95% CI, 100 MC runs)', fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(COMP_DIR / 'plots' / 'all_configs_overall_overlay.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Plot: Best vs Default macro ──
if best_config != 'A_default':
    fig, ax = plt.subplots(figsize=(10, 6))
    d = config_summaries['A_default']
    b = config_summaries[best_config]
    
    ax.plot(d['shots'], d['mean_macro'], 'o--', color='#4C72B0', linewidth=1.5,
            markersize=6, label='A_default (C=1, scale, rbf)', alpha=0.7)
    ax.fill_between(d['shots'], d['mean_macro'] - d['ci95_macro'],
                    d['mean_macro'] + d['ci95_macro'], alpha=0.1, color='#4C72B0')
    
    ax.plot(b['shots'], b['mean_macro'], 'o-',
            color=CONFIG_COLORS.get(best_config, '#E07B39'),
            linewidth=2.5, markersize=8, label=f'{best_config}')
    ax.fill_between(b['shots'], b['mean_macro'] - b['ci95_macro'],
                    b['mean_macro'] + b['ci95_macro'], alpha=0.15,
                    color=CONFIG_COLORS.get(best_config, '#E07B39'))
    
    for _, row in b.iterrows():
        ax.annotate(f"{row['mean_macro']*100:.1f}%",
                    (row['shots'], row['mean_macro']),
                    textcoords='offset points', xytext=(0, 12),
                    fontsize=9, fontweight='bold', ha='center',
                    color=CONFIG_COLORS.get(best_config, '#E07B39'))
    
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xticks(shots_grid); ax.set_ylim(0.5, 1.0)
    ax.set_xlabel('Shots per species', fontsize=12)
    ax.set_ylabel('Macro accuracy', fontsize=12)
    ax.set_title(f'Macro Accuracy: Default vs. Best ({best_config})\n(mean +/- 95% CI)', fontsize=13)
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(COMP_DIR / 'plots' / 'best_vs_default_macro.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print("Default config IS the best — no comparison plot needed.")

# ── Save comparison summary CSV ──
comp_rows = []
for config_name in all_names:
    s = config_summaries[config_name]
    p = ALL_CONFIGS.get(config_name, {})
    for _, row in s.iterrows():
        comp_rows.append({
            'config': config_name,
            'C': p.get('C', 'auto'), 'gamma': str(p.get('gamma', 'n/a')),
            'kernel': p.get('kernel', p.get('type', 'n/a')),
            'shots': int(row['shots']),
            'mean_macro': row['mean_macro'],
            'ci95_macro': row['ci95_macro'],
            'mean_overall': row['mean_overall'],
            'ci95_overall': row.get('ci95_overall', np.nan),
        })
comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv(COMP_DIR / 'analysis' / 'all_configs_summary.csv', index=False)
print(f"Comparison summary saved to {COMP_DIR / 'analysis' / 'all_configs_summary.csv'}")

## 5. Confusion Matrices

Row-normalized mean confusion matrices per config, plus side-by-side best vs default.

In [ ]:
# === Confusion matrix helpers ===

def mean_rownorm_confusion_for_K(df, K, class_order=None):
    """Row-normalize confusion per run, then average over runs."""
    dK = df[df["shots"] == K].copy()
    if class_order is None:
        classes = sorted(pd.unique(dK["species_true"]))
    else:
        classes = [c for c in class_order if c in set(dK["species_true"])]
    n = len(classes)
    if n == 0:
        raise ValueError(f"No classes present at K={K}.")

    mean_conf = np.zeros((n, n), dtype=float)
    row_counts = np.zeros(n, dtype=int)

    for run_id, dKr in dK.groupby("run_id"):
        ct = pd.crosstab(
            pd.Series(dKr["species_true"], name="true"),
            pd.Series(dKr["species_pred"], name="pred")
        )
        ct = ct.reindex(index=classes, columns=classes, fill_value=0)
        m = ct.to_numpy().astype(float)
        row_sums = m.sum(axis=1, keepdims=True)
        nonzero_rows = (row_sums[:, 0] > 0)
        if not np.any(nonzero_rows):
            continue
        m_norm = np.zeros_like(m)
        m_norm[nonzero_rows] = m[nonzero_rows] / row_sums[nonzero_rows]
        mean_conf[nonzero_rows] += m_norm[nonzero_rows]
        row_counts[nonzero_rows] += 1

    for i in range(n):
        if row_counts[i] > 0:
            mean_conf[i] /= row_counts[i]

    return classes, mean_conf, row_counts


def plot_mean_confusion(K, classes, mean_conf, outpath, title_prefix=""):
    n = len(classes)
    fig_h = max(6, 0.55 * n)
    fig_w = max(7, 0.7 * n)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(mean_conf, aspect="auto", cmap="Blues", vmin=0, vmax=1)
    y_labels = [f"{cls}  (N={SPECIES_N.get(cls, 0)})" for cls in classes]

    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.set_xlabel("Predicted species", fontsize=11)
    ax.set_ylabel("True species", fontsize=11)
    ax.set_title(f"{title_prefix}Row-normalized mean confusion — {K}-shot", fontsize=12)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.yaxis.set_major_formatter(PercentFormatter(1.0))

    for i in range(n):
        for j in range(n):
            val = mean_conf[i, j]
            text_color = "white" if val > 0.5 else "#1a1a2e"
            fontweight = "bold" if i == j else "normal"
            ax.text(j, i, f"{val*100:.0f}%",
                    ha="center", va="center", fontsize=10,
                    color=text_color, fontweight=fontweight)

    plt.tight_layout()
    plt.savefig(outpath, dpi=200, bbox_inches='tight')
    plt.show()

print("Confusion helpers loaded.")

In [ ]:
# === Per-config confusion matrices (new configs only) ===

for config_name in CONFIGS:
    df_cfg = all_config_predictions[config_name]
    plots_dir = CONFIG_DIRS[config_name] / 'plots'
    for K in SHOTS:
        classes, M, counts = mean_rownorm_confusion_for_K(df_cfg, K, class_order=CLASS_ORDER)
        plot_mean_confusion(K, classes, M,
                            plots_dir / f"confusion_mean_rownorm_K{K:02d}.png",
                            title_prefix=f"{config_name} — ")
    print(f"[{config_name}] {len(SHOTS)} confusion matrices saved")

In [ ]:
# === Side-by-side confusion: Default vs Best at max K ===

MAX_K = max(SHOTS)

if best_config != 'A_default':
    df_def = all_config_predictions['A_default']
    df_best = all_config_predictions[best_config]
    bp = ALL_CONFIGS.get(best_config, {})

    def_classes, def_cm, _ = mean_rownorm_confusion_for_K(df_def, MAX_K, class_order=CLASS_ORDER)
    best_classes, best_cm, _ = mean_rownorm_confusion_for_K(df_best, MAX_K, class_order=CLASS_ORDER)
    n = len(best_classes)

    # Build title for best config
    if bp.get('type') == 'logreg':
        best_title = f'{best_config} ({MAX_K}-shot)\nLogReg (L2, saga, balanced)'
    else:
        best_title = f'{best_config} ({MAX_K}-shot)\nC={bp.get("C","?")}, gamma={bp.get("gamma","?")}, {bp.get("kernel","?")}'

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6.5))
    y_labels = [f"{cls}  (N={SPECIES_N.get(cls, 0)})" for cls in best_classes]

    im1 = ax1.imshow(def_cm, aspect='auto', cmap='Blues', vmin=0, vmax=1)
    ax1.set_xticks(range(n)); ax1.set_yticks(range(n))
    ax1.set_xticklabels(best_classes, rotation=45, ha='right', fontsize=9)
    ax1.set_yticklabels(y_labels, fontsize=9)
    ax1.set_xlabel('Predicted', fontsize=11); ax1.set_ylabel('True', fontsize=11)
    ax1.set_title(f'A_default ({MAX_K}-shot)\nC=1, gamma=scale, rbf', fontsize=12)
    for i in range(n):
        for j in range(n):
            val = def_cm[i, j]
            ax1.text(j, i, f'{val*100:.0f}%', ha='center', va='center',
                     color='white' if val > 0.5 else '#1a1a2e', fontsize=10,
                     fontweight='bold' if i == j else 'normal')

    im2 = ax2.imshow(best_cm, aspect='auto', cmap='Blues', vmin=0, vmax=1)
    ax2.set_xticks(range(n)); ax2.set_yticks(range(n))
    ax2.set_xticklabels(best_classes, rotation=45, ha='right', fontsize=9)
    ax2.set_yticklabels([], fontsize=9)
    ax2.set_xlabel('Predicted', fontsize=11)
    ax2.set_title(best_title, fontsize=12)
    for i in range(n):
        for j in range(n):
            val = best_cm[i, j]
            ax2.text(j, i, f'{val*100:.0f}%', ha='center', va='center',
                     color='white' if val > 0.5 else '#1a1a2e', fontsize=10,
                     fontweight='bold' if i == j else 'normal')

    fig.subplots_adjust(wspace=0.05)
    cbar = fig.colorbar(im2, ax=[ax1, ax2], fraction=0.046, pad=0.04, shrink=0.8)
    cbar.ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    plt.tight_layout()
    out = COMP_DIR / 'plots' / f'best_vs_default_confusion_K{MAX_K}.png'
    plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out}")
else:
    print("Default IS the best config — no side-by-side needed.")

## 6. Summary

In [ ]:
# === Final Summary ===

print("=" * 70)
print("HYPERPARAMETER COMPARISON SUMMARY")
print("=" * 70)
print(f"\nMethod: Direct config comparison ({len(all_names)} configs x {RUNS} MC runs x {len(SHOTS)} K values)")
print(f"New configs run: {list(CONFIGS.keys())}")
print(f"Previous configs loaded: {list(PREV_CONFIGS.keys())}")
print(f"Results: {RUN_DIR}\n")

# Summary table
print(f"{'Config':<20} {'C':>5} {'gamma':>8} {'kernel':>8}  ", end="")
for K in SHOTS:
    print(f"  K={K:<2}", end="")
print()
print("-" * 110)

for config_name in sorted(all_names):
    p = ALL_CONFIGS.get(config_name, {})
    s = config_summaries[config_name]
    marker = " <-- BEST" if config_name == best_config else ""
    c_str = str(p.get('C', 'auto'))
    g_str = str(p.get('gamma', 'n/a'))
    k_str = p.get('kernel', p.get('type', 'n/a'))
    print(f"{config_name:<20} {c_str:>5} {g_str:>8} {k_str:>8}  ", end="")
    for K in SHOTS:
        val = s[s['shots'] == K]['mean_macro'].values[0] * 100
        print(f"  {val:5.1f}", end="")
    print(marker)

print("-" * 110)

bp = ALL_CONFIGS.get(best_config, {})
print(f"\nBest config: {best_config}")
if bp.get('type') == 'logreg':
    print(f"  LogisticRegression (L2, saga, balanced)")
else:
    print(f"  C={bp.get('C','?')}, gamma={bp.get('gamma','?')}, kernel={bp.get('kernel','?')}")

if best_config != 'A_default':
    d35 = config_summaries['A_default'][config_summaries['A_default']['shots']==35]['mean_macro'].values[0]
    b35 = config_summaries[best_config][config_summaries[best_config]['shots']==35]['mean_macro'].values[0]
    diff = (b35 - d35) * 100
    sign = '+' if diff >= 0 else ''
    print(f"  Improvement over default at K=35: {sign}{diff:.2f} pp")
else:
    print("  Default parameters are already optimal.")